In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(f'Tensorflow version: {tf.version.VERSION}')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: ', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))


In [ ]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub
# try:
#     import huggingface_hub
# except ImportError:
#     !pip install huggingface_hub
# try:
#     import ipywidgets
# except ImportError:
#     !pip install ipywidgets
#     !pip install --upgrade ipywidgets

# !pip install --upgrade notebook jupyterlab

from src.dataset.DroneSegDataSet import MyDataset
from src.model.U_NetModel import U_Net
from src.model.MultiU_NetModel import MultiU_Net
from src.training.TrainSession import train_session
from src.dataset.CheckDataset import check_dataset
from src.training.Criterion import criterion as criterion


In [ ]:
def print_model_params(model):
    print("Model parameter summary:")
    print("-" * 60)
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            numel = param.numel()
            total += numel
            print(f"{name:<50} {numel:>12,}")
    print("-" * 60)
    print(f"{'Total trainable parameters':<50} {total:>12,}")


In [ ]:
# 检查硬件环境
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# model = U_Net(
#     in_channel=22,
#     depth=4,
# ).to(device)
#
# model.eval()
#
# print("model parameters count")
# print_model_params(model)

# print("Model summary: ")
# print(model)


In [ ]:
from torchvision import transforms
# 加载数据集
ds_path = check_dataset()
dataset = MyDataset(
    'drone_seg_dataset/classes_dataset/classes_dataset/original_images',
    'drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
    transform=transforms.Compose([
        transforms.ToTensor(),
    ]),
    # data_enforcement=True,
)

from torch.utils.data import random_split, DataLoader
import numpy as np

train_set, test_set = random_split(dataset, [int(0.9 * len(dataset)), len(dataset) - int(0.9 * len(dataset))])
train_loader = DataLoader(dataset=train_set, batch_size=2, shuffle=True)
# test_loader = DataLoader(dataset=test_set, batch_size=2, shuffle=False)

print(f"数据集总样本数: {len(dataset)}")
print(f"训练集样本数: {len(train_set)}")
print(f"测试集样本数: {len(test_set)}")

img0, lbl0, _ = train_set.__getitem__(0)
print(f"Train set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
lbl_unique = torch.unique(lbl0)
print(f"Train set - unique labels in lbl0: {lbl_unique}")

# img0, lbl0, _ = test_set.__getitem__(0)
# print(f"Test set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
# lbl_unique = torch.unique(lbl0)
# print(f"Train set - unique labels in lbl0: {lbl_unique}")

# img0, lbl0, _ = dataset.__getitem__(0)
#
# print("图像1的形状:", img0.shape)
# print("标签1的形状:", lbl0.shape)
#
# img0_batched = img0.unsqueeze(0)
# ans0 = model(img0_batched.to(device))
# print("模型输出的形状:", ans0.shape)
#
# sample_output = torch.argmax(ans0, dim=1).detach().cpu().numpy()
# print("模型输出的唯一值 (类别索引):", np.unique(sample_output))


In [ ]:
# INIT_MODEL_PATH = "" # 在此处填入.pth文件路径以加载预训练权重，留空则从头开始训练
# if INIT_MODEL_PATH:
#     model.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=device))
#     print(f"已从 '{INIT_MODEL_PATH}' 加载模型参数。")
# else:
#     print("未指定预训练模型，从头开始训练。")

from src.dataset.DroneSegDataSet import convert_to_binary_label

# --- 定义损失函数和优化器 ---

# 损失函数使用从 src.training.Criterion 导入的 criterion，这个函数是二元交叉熵、Dice Loss 和 mIoU 损失的组合，适用于二分类任务。

class_idx_list = [0]
# class_idx_list = [0, 1, 2, 3, 4]
depth_list = [3]
# depth_list = [2, 3, 4]

for depth in depth_list:
    for class_idx in class_idx_list:
        print(f"训练模型 - 类别索引: {class_idx}, 深度: {depth}")
        model = U_Net(
            in_channel=22,
            depth=depth,
            depthwise_separable=True,
        ).to(device)

        def convert(labels: torch.Tensor) -> torch.Tensor:
            bin_label = convert_to_binary_label(labels, class_idx) # 格式为 [B, H, W]，值为0或1
            # 需要增加 通道C 维度，变为 [B, 2, H, W] 的 one-hot 编码，以适配 BCEWithLogitsLoss 的输入要求
            labels_one_hot = torch.stack([
                1 - bin_label,  # 通道0: 背景 (原标签为0的地方是1，为1的地方是0)
                bin_label       # 通道1: 前景 (原标签为1的地方是1，为0的地方是0)
            ], dim=1).float()
            return labels_one_hot.to(device)

        train_session(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            title=f"U-Net Class {class_idx} Depth {depth}",
            device=device,
            label_transform=convert,  # 将标签转换为二分类格式
        )
